# 02 — Embedding Analysis

Embed the sample chunks, reduce dimensionality with PCA and t-SNE, and visualise the semantic space.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

from src.ingestion.loader import DocumentLoader
from src.ingestion.chunker import TextChunker
from src.embeddings.embedder import Embedder

## 1. Load and Chunk Documents

In [ ]:
loader = DocumentLoader()
chunker = TextChunker(chunk_size=400, chunk_overlap=40)

documents = loader.load_directory('../data/sample_docs')
chunks = chunker.chunk_documents(documents)

texts = [c['content'] for c in chunks]
print(f'Chunks to embed: {len(texts)}')

## 2. Generate Embeddings

In [ ]:
embedder = Embedder()
print(f'Model     : {embedder.model_name}')
print(f'Device    : {embedder.device}')
print(f'Embed dim : {embedder.embedding_dim}')

embeddings = embedder.embed_texts(texts)
print(f'Embeddings shape: {embeddings.shape}')

## 3. PCA — 2D Projection

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(embeddings)
print(f'PCA explained variance ratio: {pca.explained_variance_ratio_.sum():.1%}')

fig, ax = plt.subplots(figsize=(9, 6))
colors = cm.tab10(np.linspace(0, 1, len(chunks)))
sc = ax.scatter(pca_coords[:, 0], pca_coords[:, 1], c=colors, s=80, alpha=0.85, edgecolors='white', linewidth=0.5)

for i, (x, y) in enumerate(pca_coords):
    label = texts[i][:30].replace('\n', ' ') + '…'
    ax.annotate(label, (x, y), fontsize=6, alpha=0.7, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA Projection of Chunk Embeddings (all-MiniLM-L6-v2)', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. t-SNE — 2D Projection

In [ ]:
perplexity = min(5, max(1, len(texts) - 1))
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
tsne_coords = tsne.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(tsne_coords[:, 0], tsne_coords[:, 1], c=colors, s=80, alpha=0.85, edgecolors='white', linewidth=0.5)

for i, (x, y) in enumerate(tsne_coords):
    label = texts[i][:30].replace('\n', ' ') + '…'
    ax.annotate(label, (x, y), fontsize=6, alpha=0.7, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE Projection of Chunk Embeddings', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Cosine Similarity Heatmap

In [ ]:
sim_matrix = cosine_similarity(embeddings)
short_labels = [t[:25].replace('\n', ' ') + '…' for t in texts]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(short_labels, fontsize=7)

# Annotate cells
for i in range(len(texts)):
    for j in range(len(texts)):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center', fontsize=6,
                color='black' if sim_matrix[i, j] < 0.7 else 'white')

ax.set_title('Chunk-to-Chunk Cosine Similarity Matrix', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print(f'Average off-diagonal similarity: {(sim_matrix.sum() - np.trace(sim_matrix)) / (len(texts)**2 - len(texts)):.4f}')

## 6. Query Similarity Demo

In [ ]:
queries = [
    'What is machine learning?',
    'Explain neural networks.',
    'What are the applications of AI?',
]

for query in queries:
    q_emb = embedder.embed_query(query).reshape(1, -1)
    sims = cosine_similarity(q_emb, embeddings)[0]
    top_idx = sims.argsort()[::-1][:3]
    print(f'Query: "{query}"')
    for rank, idx in enumerate(top_idx, 1):
        preview = texts[idx][:80].replace('\n', ' ')
        print(f'  [{rank}] sim={sims[idx]:.4f}  {preview}…')
    print()